# AgriDrone Matrix Runner — Kaggle Edition

Runs the same 54-cell ablation matrix as the Colab notebook but on Kaggle Notebooks:
- **Free GPU**: T4 × 2 or P100 (pick in right panel → *Accelerator*)
- **Weekly budget**: ~30 hours (vs Colab free's ~4h/day)
- **Session length**: up to 12h uninterrupted
- **Persistent output**: `/kaggle/working/` survives commit, can be downloaded

## One-time setup (before first run)
1. **Settings → Accelerator → GPU T4 x2** (or P100)
2. **Settings → Internet → On** (required to clone repo + download datasets)
3. **Add-ons → Secrets → Add secret** named `KAGGLE_JSON` (paste your kaggle.json contents) — optional, only if you hit private-dataset auth issues; Kaggle Notebooks already have Kaggle API configured for the logged-in user.

## Resume
`per_run.jsonl` is written to `/kaggle/working/results_v2/matrix/<run_id>/` and preserved in the notebook's output on **Save Version → Quick Save**. If the session ends, create a new version, set its input to the previous version's output, and the runner will skip completed cells.

## 1. Clone repo

In [ ]:
import os, subprocess, sys
REPO = 'https://github.com/Ashut0sh-mishra/agri-drone.git'
WORK = '/kaggle/working/agri-drone'
if not os.path.exists(WORK):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, WORK])
else:
    subprocess.check_call(['git', '-C', WORK, 'pull', '--ff-only'])
os.chdir(WORK)
print('HEAD:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode().strip())

## 2. Install dependencies (Kaggle already has torch/timm/kaggle — we only add the small extras)

In [ ]:
%pip install -q ultralytics==8.3.0 omegaconf pyyaml
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

## 3. Configure Kaggle CLI
Kaggle Notebooks are already authenticated for the *notebook owner's* Kaggle account — `kaggle datasets download` works out of the box. Nothing to do unless you pasted a `KAGGLE_JSON` secret, in which case the next cell installs it.

In [ ]:
import os, json, pathlib
try:
    from kaggle_secrets import UserSecretsClient
    secret = UserSecretsClient().get_secret('KAGGLE_JSON')
    kdir = pathlib.Path.home() / '.kaggle'
    kdir.mkdir(exist_ok=True)
    (kdir / 'kaggle.json').write_text(secret)
    (kdir / 'kaggle.json').chmod(0o600)
    print('Installed KAGGLE_JSON secret.')
except Exception as e:
    print('No KAGGLE_JSON secret (OK — notebook owner auth will be used):', e)
# Smoke-test
import subprocess; print(subprocess.check_output(['kaggle', '--version']).decode().strip())

## 4. Pick the matrix config
- `smoke.yaml` → 6 cells, ~15 min (sanity check)
- `large.yaml` → 54 cells, ~6–10h (full ablation)

In [ ]:
CONFIG = 'configs/matrix/large.yaml'  # or 'configs/matrix/smoke.yaml'
OUT_DIR = '/kaggle/working/results_v2/matrix'
os.makedirs(OUT_DIR, exist_ok=True)
print('config:', CONFIG); print('out_dir:', OUT_DIR)

## 5. Run the matrix (streams live output, auto-resumes)
Outer retry loop handles transient failures (Kaggle download hiccups, etc.). Inner resume loop in `run_matrix.py` skips cells whose `status == 'ok'` in `per_run.jsonl`.

In [ ]:
import subprocess, sys, time
MAX_OUTER_ATTEMPTS = 6
for attempt in range(1, MAX_OUTER_ATTEMPTS + 1):
    print(f'\n===== outer attempt {attempt}/{MAX_OUTER_ATTEMPTS} =====')
    proc = subprocess.Popen(
        [sys.executable, 'evaluate/matrix/run_matrix.py',
         '--config', CONFIG, '--out-dir', OUT_DIR],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        bufsize=1, text=True, cwd=WORK,
    )
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    print(f'\n[outer] exit={rc}')
    if rc == 0:
        print('[outer] run complete.')
        break
    print('[outer] retrying in 30s…'); time.sleep(30)

## 6. Quick summary of what landed in per_run.jsonl

In [ ]:
import json, glob, collections
files = glob.glob(f'{OUT_DIR}/*/per_run.jsonl')
print('per_run files:', files)
for f in files:
    rows = [json.loads(l) for l in open(f) if l.strip()]
    c = collections.Counter(r.get('status') for r in rows)
    print(f'{f}: total={len(rows)} {dict(c)}')
    if rows:
        last = rows[-1]
        print('  last:', last.get('backbone'), last.get('dataset'), 'top1=', (last.get('metrics') or {}).get('top1'))

## 7. Save output
Click **Save Version** (top right) → *Save & Run All (Commit)* to snapshot `/kaggle/working/` as a permanent notebook version you can re-use as input for a resume run.